In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.measure import label
import torch
from torch import Tensor
from torchvision.ops.boxes import box_area
import math
import re
import base64
import os
import io
from tqdm import tqdm
import random
import cv2
from PIL import Image, ImageDraw
from pycocotools import mask as maskUtils

In [2]:
def get_random_nonzero_point(mask):
    """
    在mask的非零区域随机选取一个点并返回其坐标
    
    参数:
    mask: numpy数组，包含0和非0值
    
    返回:
    tuple: 随机选取的非零点的坐标
    """
    # 获取所有非零元素的坐标
    nonzero_coords = np.nonzero(mask)
    
    # 检查是否存在非零元素
    if len(nonzero_coords[0]) == 0:
        raise ValueError("Mask中不存在非零元素")
    
    # 随机选择一个非零点的索引
    random_idx = np.random.randint(0, len(nonzero_coords[0]))
    
    # 构建坐标元组
    point_coord = tuple(nonzero_coords[dim][random_idx].tolist() for dim in range(mask.ndim))
    
    return point_coord

In [3]:
from refer import REFER

In [4]:
data_path = r"F:\Datasets\COCO2014\refer_seg\images\mscoco"
img_path = os.path.join(data_path,"images","train2014")
task = "refcoco"
refer = REFER(data_path, task, "unc")

loading dataset refcoco into memory...
creating index...
index created.
DONE (t=6.60s)


In [5]:
len(refer.data['annotations'])

196771

In [6]:
instance = json.load(open(r"F:\Datasets\COCO2014\annotations\instances_train2014.json"))

In [7]:
len(instance['images'])

82783

In [13]:
image_id_dict = {item['id']:item for item in instance['images']}

In [14]:
ann_dict = {item['id']:[] for item in instance['images']}

In [15]:
final_data = {}

In [16]:
for ann in tqdm(instance['annotations']):
    # if ann['image_id'] not in ann_dict.keys():
    #     continue
    image = image_id_dict[ann['image_id']]
    rle = maskUtils.frPyObjects(ann['segmentation'], image['height'],image['width'])
    if isinstance(rle,list):
        rle = rle[0]
    # break
    m = maskUtils.decode(rle)
    if len(m.shape)>2:
        m = np.sum(
            m, axis=2
        )  # sometimes there are multiple binary map (corresponding to multiple segs)
    m = m.astype(np.uint8)  # convert to np.uint8
    try:
        random_point = get_random_nonzero_point(m)
    except:
        continue
    bbox = ann['bbox']
    rle['counts'] = rle['counts'].decode('utf-8')
    obj = {
        "bbox":bbox,
        "area":ann['area'],
        "segmentation":rle,
        "crop_box" : [0,0,image['width'],image['height']],
        "id":ann['id'],
        "stability_score":1,
        "point_coords":[list(random_point)[::-1]]
    }
    ann_dict[ann['image_id']].append(obj)
    pass
    # break

100%|█████████████████████████████████████████████████████████████████████████| 604907/604907 [12:44<00:00, 791.14it/s]


In [17]:
for key in ann_dict.keys():
    img_info = image_id_dict[key]
    final_data[key] = {
        "image":{
            'image_id': img_info['id'], 'width': img_info['width'], 'height': img_info['height'], 'file_name': img_info['file_name']
        },
        "annotations":ann_dict[key]
    }
    # break

In [18]:
import shutil
import json
import os
from pathlib import Path

def copy_image_and_save_dict(image_path, data_dict, output_dir="dir2"):
    """
    将图像复制到指定目录，并将字典保存为同名的JSON文件。
    
    参数:
    image_path (str): 源图像文件的路径
    data_dict (dict): 要保存的字典数据
    output_dir (str): 目标目录，默认为'dir2'
    """
    # 确保输出目录存在
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # 获取图像文件名（不含路径）
    image_filename = os.path.basename(image_path)
    
    # 构建目标图像路径
    dest_image_path = os.path.join(output_dir, image_filename)
    
    # # 复制图像文件
    # shutil.copy2(image_path, dest_image_path)
    # print(f"图像已复制到: {dest_image_path}")
    
    # 构建JSON文件路径（相同名称，不同后缀）
    json_filename = os.path.splitext(image_filename)[0] + ".json"
    json_path = os.path.join(output_dir, json_filename)
    
    # 将字典保存为JSON文件
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data_dict, f, indent=4, ensure_ascii=False)
    # print(f"字典已保存为JSON: {json_path}")

In [19]:
for key,item in final_data.items():
    image_path = os.path.join(img_path,item['image']['file_name'])
    copy_image_and_save_dict(image_path,item,output_dir="coco_sa1b")
    # break
